## Document term matrices for each of the 5 datasets

In [ ]:
import json
import pickle
import numpy as np
import os
from scipy.sparse import save_npz
from sklearn.feature_extraction.text import CountVectorizer

Using CountVectorizer to build DTMs in one step and creating vocabulary files to do the mapping between entries and terms - ignoring overly common or overly rare tokens - calculating sparsity of DTMs

In [ ]:
#setting up folder paths 
root_folder = "../articles_raw_proc_data"
matrices_folder = "articles_raw_proc_data/matrices"
models_folder = "models"
train_file_path = root_folder + "/processed_data/guardian_train.json"

#Load articles from train set
train_file = open(train_file_path, "r", encoding="utf-8")
train = json.load(train_file)
train_file.close()
print("total training articles loaded: " + str(len(train)))

#Train set contains fields for each of the datasets
fields_to_process = ["text_all", "text_all_lemma", "text_nouns", "text_nouns_lemma", "text_content_stem"]

#looping over each field one at a time
for field in fields_to_process:
    #collecting the concatenated text for this field from every article
    texts_for_this_field = []
    for article in train:
        one_text = article[field]
        texts_for_this_field.append(one_text)

    #Building vectorizer
    vectorizer = CountVectorizer(min_df=2, max_df=0.95, max_features=None)

    #Applying vectorizer to dataset to obtain document-term matrix
    dtm = vectorizer.fit_transform(texts_for_this_field)

    #Separate file for vocabulary map columns of DTM to actual words
    vocab = vectorizer.get_feature_names_out()

    #building save paths for the matrix and vocabulary files
    dtm_save_path   = matrices_folder + "/dtm_" + field + ".npz"
    vocab_save_path = matrices_folder + "/vocab_" + field + ".npy"
    vec_save_path   = models_folder   + "/vectorizer_" + field + ".pkl"

    #saving the sparse matrix as .npz (a compressed format)
    save_npz(dtm_save_path, dtm)
    print("  saved dtm to: " + dtm_save_path)

    #saving the vocabulary array with numpy
    np.save(vocab_save_path, vocab)
    print("  saved vocab to: " + vocab_save_path)

    #saving the fitted vectorizer with pickle so we can reload it later
    #we need to open in binary write mode "wb" for pickle — not sure why text mode doesn't work
    vec_file = open(vec_save_path, "wb")
    pickle.dump(vectorizer, vec_file)
    vec_file.close()
    print("  saved vectorizer to: " + vec_save_path)

    #pulling out shape info for the summary print
    #dtm.shape is a tuple like (num_docs, num_words)
    num_docs  = dtm.shape[0]
    num_words = dtm.shape[1]

    #calculating sparsity 
    #nnz = number of non-zero entries in the sparse matrix
    total_cells   = num_docs * num_words
    nonzero_cells = dtm.nnz
    zero_cells    = total_cells - nonzero_cells

    sparsity_fraction = zero_cells / total_cells
    sparsity_percent = round(sparsity_fraction * 100, 4)
    sparsity_string  = str(sparsity_percent) + "%"

    #printing the summary line for this field
    print("  field=" + field + "  docs=" + str(num_docs) + "  vocab=" + str(num_words) + "  sparsity=" + sparsity_string)
    print("  ----")

#Confirming save
print("saved dtms to folder:        " + matrices_folder)
print("saved vectorizers to folder: " + models_folder)


## NNSVD-LRC initialization 



Scripts for NNSVD-LRC are ported from MATLAB code by Claude Sonet 

In [ ]:
import numpy as np
from pathlib import Path
from scipy.sparse import load_npz
from prettytable import PrettyTable
from itertools import combinations
import sys
sys.path.insert(0, str(Path.cwd()))
from nnsvdlrc import nnsvdlrc
from eval_h_stability import top_n_indices, pairwise_ats

MATRICES = Path("articles_raw_proc_data/matrices")
dtm = load_npz(MATRICES / "dtm_text_all_lemma.npz").astype(float)

r = 14
n_runs = 1
top = 25

Hs, Ws, errors, iters, all_rankings = [], [], [], [], []
for i in range(n_runs):
    W, H, _, _, e = nnsvdlrc(dtm, r)
    Hs.append(H)
    Ws.append(W)
    errors.append(e[-1])
    iters.append(len(e))
    all_rankings.append(top_n_indices(H, top))
    print(f"  Run {i+1:>2}/{n_runs}  final_error={e[-1]:.6f}  iters={len(e)}")

MODELS = Path("models")
MODELS.mkdir(exist_ok=True)
best_idx = int(np.argmin(errors))
np.save(MODELS / "H_init_text_all_lemma14.npy", Hs[best_idx])
np.save(MODELS / "W_init_text_all_lemma14.npy", Ws[best_idx])

print()
tab = PrettyTable(["pair", "ATS", "per-topic matches"])
tab.align["per-topic matches"] = "l"
tab.align["ATS"] = "r"
for i, j in combinations(range(n_runs), 2):
    score, matches, S = pairwise_ats(all_rankings[i], all_rankings[j])
    match_str = "  ".join(f"T{r1+1}->T{r2+1}({S[r1,r2]:.2f})" for r1, r2 in matches)
    tab.add_row([f"{i+1} vs {j+1}", f"{score:.4f}", match_str])
print(f"All-pairs ATS  (top {top} terms, r={r})")
print(tab)

Spot-checking H after NNSVD-LRC

In [ ]:
import pickle
import numpy as np
import plotly.graph_objects as go
from pathlib import Path

MODELS = Path("models")

with open(MODELS / "vectorizer_text_all_lemma.pkl", "rb") as f:
    vectorizer = pickle.load(f)

vocab = np.array(vectorizer.get_feature_names_out())
H = np.load(MODELS / "H_init_text_all_lemma.npy")

n_top = 25
n_topics = H.shape[0]

cols = []
for topic_idx in range(n_topics):
    top_idx = np.argsort(H[topic_idx])[::-1][:n_top]
    terms = vocab[top_idx]
    weights = H[topic_idx][top_idx]
    # pad terms to same length so loadings align
    max_len = max(len(t) for t in terms)
    combined = [f"{t:<{max_len}}   {w:.3f}" for t, w in zip(terms, weights)]
    cols.append(combined)

fig = go.Figure(data=[go.Table(
    header=dict(
        values=[f"Topic {i}" for i in range(n_topics)],
        fill_color='steelblue',
        font=dict(color='white', size=13, family='monospace'),
        align='center'
    ),
    cells=dict(
        values=cols,
        align='left',
        fill_color='white',
        font=dict(family='monospace', size=12)
    )
)])

fig.update_layout(title='Top 25 terms per NMF topic (H_init)', height=800)
fig.show()

## NMF stability (KL + multiplicative updates)

Each run re-loads the W and H matrices initizalized previously by NNSVD-LRC, then fits NMF with those as custom init.
ATS is computed on the final H matrices across all pairs of runs.

In [ ]:
import numpy as np
from pathlib import Path
from scipy.sparse import load_npz, issparse
from sklearn.decomposition import NMF
from prettytable import PrettyTable
from itertools import combinations
import sys
sys.path.insert(0, str(Path.cwd()))
from eval_h_stability import top_n_indices, pairwise_ats

MATRICES = Path("articles_raw_proc_data/matrices")
MODELS = Path("models")
dtm = load_npz(MATRICES / "dtm_text_all_lemma.npz").astype(float)

r = 14
n_runs = 2
top = 25

W_init = np.load(MODELS / "W_init_text_all_lemma14.npy")
H_init = np.load(MODELS / "H_init_text_all_lemma14.npy")

def kl_div(X, W, H):
    WH = W @ H
    WH = np.maximum(WH, np.finfo(float).eps)
    if issparse(X):
        rows, cols = X.nonzero()
        x_vals = np.asarray(X[rows, cols]).ravel()
        wh_vals = WH[rows, cols]
    else:
        mask = X > 0
        x_vals, wh_vals = X[mask], WH[mask]
    return float(np.sum(x_vals * np.log(x_vals / wh_vals) - x_vals) + WH.sum())

Ws, Hs, kl_errors, frob_errors, iters, all_rankings = [], [], [], [], [], []
for i in range(n_runs):
    nmf = NMF(
        n_components=r,
        init="custom",
        solver="mu",
        beta_loss="kullback-leibler",
        max_iter=500,
        random_state=123,
    )
    W = nmf.fit_transform(dtm, W=W_init.copy(), H=H_init.copy())
    H = nmf.components_
    Ws.append(W)
    Hs.append(H)
    kl_errors.append(kl_div(dtm, W, H))
    frob_errors.append(nmf.reconstruction_err_)
    iters.append(nmf.n_iter_)
    all_rankings.append(top_n_indices(H, top))
    print(f"  Run {i+1:>2}/{n_runs}  KL={kl_errors[-1]:.4f}  "
          f"Frob={frob_errors[-1]:.4f}  iters={nmf.n_iter_}")

# --- select best run by mean ATS ---
n = len(all_rankings)
mean_ats = np.zeros(n)
for i in range(n):
    scores = [pairwise_ats(all_rankings[i], all_rankings[j])[0]
              for j in range(n) if j != i]
    mean_ats[i] = np.mean(scores)
best_score = mean_ats.max()
candidates = np.where(mean_ats == best_score)[0]
best_run = int(np.random.choice(candidates))
print(f"\nBest run: {best_run + 1}  (mean ATS = {best_score:.4f})")

np.save(MODELS / "H_best_text_all_lemma14.npy", Hs[best_run])
np.save(MODELS / "W_best_text_all_lemma14.npy", Ws[best_run])

print()
tab = PrettyTable(["pair", "ATS", "per-topic matches"])
tab.align["per-topic matches"] = "l"
tab.align["ATS"] = "r"
for i, j in combinations(range(n_runs), 2):
    score, matches, S = pairwise_ats(all_rankings[i], all_rankings[j])
    match_str = "  ".join(f"T{r1+1}->T{r2+1}({S[r1,r2]:.2f})" for r1, r2 in matches)
    tab.add_row([f"{i+1} vs {j+1}", f"{score:.4f}", match_str])
print(f"All-pairs ATS after NMF (KL, mu)  (top {top} terms, r={r}, dtm_text_all_lemma)")
print(tab)

## Topic-term association for full, lemmatized dataset — visual inspection of top 25 terms in H

Best run selected by highest mean ATS across all other runs. 

In [ ]:
import numpy as np
import plotly.graph_objects as go
from pathlib import Path
import pickle

MODELS = Path("models")

with open(MODELS / "vectorizer_text_all_lemma.pkl", "rb") as f:
    vectorizer = pickle.load(f)
vocab = np.array(vectorizer.get_feature_names_out())

H_best = np.load(MODELS / "H_best_text_all_lemma14.npy")
n_topics = H_best.shape[0]
top_display = 25

cols = []
for topic_idx in range(n_topics):
    top_idx = np.argsort(H_best[topic_idx])[::-1][:top_display]
    terms = vocab[top_idx]
    weights = H_best[topic_idx][top_idx]
    max_len = max(len(t) for t in terms)
    combined = [f"{t:<{max_len}}   {w:.3f}" for t, w in zip(terms, weights)]
    cols.append(combined)

fig = go.Figure(data=[go.Table(
    header=dict(
        values=[f"Topic {i}" for i in range(n_topics)],
        fill_color='steelblue',
        font=dict(color='white', size=13, family='monospace'),
        align='center'
    ),
    cells=dict(
        values=cols,
        align='left',
        fill_color='white',
        font=dict(family='monospace', size=12)
    )
)])

fig.update_layout(title='Top 25 terms per NMF topic (H_best14)', height=800)
fig.show()

## NPMI

Number of topics are now varied between 7 and 14 and evaluated using NPMI for topic of interest in H for N = 25

In [ ]:
import numpy as np
from pathlib import Path
from scipy.sparse import load_npz
from prettytable import PrettyTable
from itertools import combinations

MATRICES = Path("articles_raw_proc_data/matrices")
MODELS   = Path("models")

dtm   = load_npz(MATRICES / "dtm_text_all_lemma.npz").astype(float)
vocab = np.load(MATRICES / "vocab_text_all_lemma.npy", allow_pickle=True)

D = dtm.shape[0]

RANK_CONFIG = [
    (7,  "H_best_text_all_lemma.npy",    2),
    (8,  "H_best_text_all_lemma8.npy",   2),
    (9,  "H_best_text_all_lemma9.npy",   2),
    (10, "H_best_text_all_lemma10.npy",  1),
    (11, "H_best_text_all_lemma11.npy",  1),
    (12, "H_best_text_all_lemma12.npy",  1),
    (13, "H_best_text_all_lemma13.npy",  2),
    (14, "H_best_text_all_lemma14.npy",  2),
]

TOP_N = 25

def doc_cooc_counts(dtm, col_i, col_j):
    """Number of documents containing both word i and word j."""
    ci = np.asarray(dtm[:, col_i].todense()).ravel() > 0
    cj = np.asarray(dtm[:, col_j].todense()).ravel() > 0
    return int(np.sum(ci & cj))

def marginal_count(dtm, col_i):
    """Number of documents containing word i."""
    return int(np.asarray(dtm[:, col_i].todense()).ravel().astype(bool).sum())

def npmi_pair(dtm, i, j, D):
    p_i  = marginal_count(dtm, i)  / D
    p_j  = marginal_count(dtm, j)  / D
    p_ij = doc_cooc_counts(dtm, i, j) / D
    eps  = 1e-12
    if p_ij < eps:
        return -1.0
    pmi = np.log(p_ij / (p_i * p_j + eps))
    npmi = pmi / (-np.log(p_ij + eps))
    return float(npmi)

def avg_pairwise_npmi(dtm, word_indices, D):
    pairs = list(combinations(word_indices, 2))
    if not pairs:
        return float("nan")
    scores = [npmi_pair(dtm, i, j, D) for i, j in pairs]
    return float(np.mean(scores))

def loading_coverage(h_row, top_indices):
    total = h_row.sum()
    if total == 0:
        return 0.0
    return float(h_row[top_indices].sum() / total)

results = []
npmi_by_rank = {}

for rank, fname, topic_idx in RANK_CONFIG:
    H = np.load(MODELS / fname)
    h_row      = H[topic_idx]
    top_idx    = np.argsort(h_row)[::-1][:TOP_N]
    top_words  = vocab[top_idx].tolist()
    coverage   = loading_coverage(h_row, top_idx)
    avg_npmi   = avg_pairwise_npmi(dtm, top_idx.tolist(), D)

    results.append({
        "rank":      rank,
        "topic_idx": topic_idx,
        "npmi":      avg_npmi,
        "coverage":  coverage,
        "top_words": top_words,
    })
    npmi_by_rank[rank] = avg_npmi
    print(f"r={rank}  topic={topic_idx}  NPMI={avg_npmi:.4f}  coverage={coverage:.3f}")

print()
tab = PrettyTable(["r", "topic", "avg NPMI", "coverage", f"top-{TOP_N} words (first 10)"])
tab.align[f"top-{TOP_N} words (first 10)"] = "l"
tab.float_format = ".4"
for res in results:
    tab.add_row([
        res["rank"],
        res["topic_idx"],
        f"{res['npmi']:.4f}",
        f"{res['coverage']:.3f}",
        ", ".join(res["top_words"][:10]),
    ])
print(tab)

np.save(MODELS / "npmi_rank_trajectory.npy", npmi_by_rank)
print("\nSaved NPMI trajectory to models/npmi_rank_trajectory.npy")

## Selecting terms based on 'lift' measure

Relevance scores of N = 25 terms for topic of interest across different values for lambda and across all other topics

In [ ]:
H = np.load("../NMF_code/models/H_lemma_r13.npy")
top_n = 25

for k in range(H.shape[0]):
    top_idx = np.argsort(H[k])[::-1][:top_n]
    terms   = [vocab[i] for i in top_idx]
    print(f"Topic {k}: {', '.join(terms)}")

Relevance scores of N = 40 terms for sibling topics across different values for lambda 

In [ ]:
# ── Step 1: row-normalize each topic independently ───────────────────────────
H = np.load("../NMF_code/models/H_lemma_r13.npy")
k        = 1
siblings = [3, 5, 7]

phi_k        = H[k] / H[k].sum()                          # typicality
phi_siblings = H[siblings].sum(axis=0)                     # aggregate sibling mass
phi_sib_norm = phi_siblings / phi_siblings.sum()           # normalize to distribution

# ── Step 2: min-max scale both components to [0, 1] ─────────────────────────
def minmax(x):
    return (x - x.min()) / (x.max() - x.min())

phi_k_scaled   = minmax(phi_k)
phi_sib_scaled = minmax(phi_sib_norm)

# ── Step 3: lambda sweep ─────────────────────────────────────────────────────
# score(w) = lambda * typicality + (1 - lambda) * penalize_siblings
# high sibling mass → low score, so we subtract
top_n   = 40
lambdas = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

rows = {}
for lam in lambdas:
    score = (1 - lam) * phi_k_scaled - lam * phi_sib_scaled
    top_idx = np.argsort(score)[::-1][:top_n]
    rows[f"λ={lam}"] = [vocab[i] for i in top_idx]

df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
import plotly.graph_objects as go

# ── Step 1: row-normalize each topic independently ───────────────────────────
H = np.load("../NMF_code/models/H_lemma_r13.npy")
k        = 1
siblings = [3, 5, 7]

phi_k        = H[k] / H[k].sum()
phi_siblings = H[siblings].sum(axis=0)
phi_sib_norm = phi_siblings / phi_siblings.sum()

# ── Step 2: min-max scale both components to [0, 1] ─────────────────────────
def minmax(x):
    return (x - x.min()) / (x.max() - x.min())

phi_k_scaled   = minmax(phi_k)
phi_sib_scaled = minmax(phi_sib_norm)

# ── Step 3: lambda sweep ─────────────────────────────────────────────────────
top_n   = 40
lambdas = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

rows     = {}
idx_rows = {}  # store indices for loading lookup
for lam in lambdas:
    score   = (1 - lam) * phi_k_scaled - lam * phi_sib_scaled
    top_idx = np.argsort(score)[::-1][:top_n]
    rows[f"λ={lam}"]     = [vocab[i] for i in top_idx]
    idx_rows[f"λ={lam}"] = top_idx

df = pd.DataFrame(rows)

# ── Step 4: Plotly table — terms ─────────────────────────────────────────────
header_vals = list(df.columns)
cell_vals   = [df[col].tolist() for col in df.columns]

fig_terms = go.Figure(data=[go.Table(
    header=dict(
        values=header_vals,
        fill_color="#1f3b5e",
        font=dict(color="white", size=11),
        align="center"
    ),
    cells=dict(
        values=cell_vals,
        fill_color=[["#f0f4f8" if i % 2 == 0 else "white"
                     for i in range(top_n)]],
        font=dict(size=10),
        align="center",
        height=22
    )
)])
fig_terms.update_layout(
    title=f"Top-{top_n} terms by λ (topic k={k} vs siblings {siblings})",
    margin=dict(l=20, r=20, t=50, b=20)
)
fig_terms.show()

# ── Step 4: Plotly table — terms + loadings in same cell ─────────────────────
rows_combined = {}
for lam in lambdas:
    score   = (1 - lam) * phi_k_scaled - lam * phi_sib_scaled
    top_idx = np.argsort(score)[::-1][:top_n]
    rows_combined[f"λ={lam}"] = [
        f"{vocab[i]}  ({phi_k[i]:.4f})" for i in top_idx
    ]

df_combined = pd.DataFrame(rows_combined)

fig_terms = go.Figure(data=[go.Table(
    header=dict(
        values=list(df_combined.columns),
        fill_color="#1f3b5e",
        font=dict(color="white", size=11),
        align="center"
    ),
    cells=dict(
        values=[df_combined[col].tolist() for col in df_combined.columns],
        fill_color=[["#f0f4f8" if i % 2 == 0 else "white"
                     for i in range(top_n)]],
        font=dict(size=10),
        align="center",
        height=22
    )
)])
fig_terms.update_layout(
    title=f"Top-{top_n} terms by λ (topic k={k} vs siblings {siblings})",
    margin=dict(l=20, r=20, t=50, b=20)
)

fig_terms.write_image("lambda_terms.png", width=2400, height=1200, scale=2)
fig_terms.show()